# HE²AT synthetic southern Africa: a first run in R

Every value in this dataset is simulated. Results are about methods, not about real people or places. Read `DATA_USE.md` in the repo before you start.

Run each cell in turn with **Shift + Enter**. This notebook uses the R runtime (Runtime > Change runtime type > R).

## 1. Get the repo

The repo is public, so Colab can clone it directly. No GitHub account or token is needed.

In [ ]:
repo <- "he2at-synthetic-southern-africa"
if (!dir.exists(repo)) {
  system(sprintf("git clone -q https://github.com/Wits-Planetary-Health-Research/%s.git", repo))
}
if (dir.exists(repo)) setwd(repo)
list.files()

Packages. `data.table` reads the zipped CSVs quickly and `dlnm` fits distributed lag models. The first install takes a minute or two.

In [ ]:
need <- c("data.table", "R.utils", "dlnm")
new <- need[!sapply(need, requireNamespace, quietly = TRUE)]
if (length(new)) install.packages(new, quiet = TRUE)
library(data.table)
library(dlnm)

## 2. Load the pregnancies

In [ ]:
p <- fread("data/participants_wide.csv.gz")      # master_long pivoted, one row per pregnancy
loc <- fread("data/locations.csv")                # LOCATION ID to district
p <- merge(p, loc[, .(Loc_id, District, urban_rural)], by.x = "LOCATION ID", by.y = "Loc_id")
dim(p)

The file uses the HEAT Master Codebook names and yes/no codes. Make short 0/1 versions for the analysis.

In [ ]:
yes <- function(x) as.integer(x == "yes")
p[, `:=`(id = `PT ID`, district_id = District,
         lmp_date = as.IDate(lnmp), birth_date = as.IDate(dob), ga_days = gest_age_delivery_days,
         preterm = yes(prem_birth), very_preterm = yes(very_prem_birth), stillbirth = yes(stillbirth),
         low_birth_weight = yes(low_birth_weight), ses_grade = ses_quintile,
         hiv_positive = as.integer(hiv_status == "positive"), anaemia = yes(anaemia),
         hypertensive_disorder = yes(hypertension_preg), outdoor_work_pregnancy = yes(outdoor_work_preg),
         infant_sex = tolower(sex))]

In [ ]:
head(p)

## 3. A first look at the outcomes

In [ ]:
cat(format(nrow(p), big.mark = ","), "pregnancies in", uniqueN(p$district_id), "districts\n")
cat("Births from", format(min(p$birth_date)), "to", format(max(p$birth_date)), "\n")
round(colMeans(p[, .(preterm, very_preterm, stillbirth, low_birth_weight)]), 3)

In [ ]:
p[, .(preterm = round(mean(preterm), 3), n = .N), by = country][order(country)]

In [ ]:
p[, .(preterm = round(mean(preterm), 3), n = .N), by = outdoor_work_pregnancy]

## 4. Map the preterm share by district

In [ ]:
gj <- jsonlite::fromJSON("data/districts.geojson", simplifyVector = FALSE)
rate <- p[, .(r = 100 * mean(preterm)), by = district_id]
setkey(rate, district_id)
greys <- gray.colors(100, start = 0.95, end = 0.15)
rng <- range(rate$r)

options(repr.plot.width = 6, repr.plot.height = 6)
par(bg = "white")
plot(NA, xlim = c(11, 41), ylim = c(-35, -8), asp = 1, xlab = "Longitude", ylab = "Latitude")
for (f in gj$features) {
  xy <- do.call(rbind, lapply(f$geometry$coordinates[[1]], unlist))
  v <- rate[f$properties$district_id, r]
  polygon(xy, col = greys[1 + round(99 * (v - rng[1]) / diff(rng))], border = "white", lwd = 0.3)
}
title("Preterm births (%), light low, dark high")

## 5. The weather: weekly district temperature from the gridded product

In [ ]:
ex <- fread("data/district_week_exposure_gridded.csv")
ex[, wk := as.integer(week_start - as.IDate("2100-01-01")) %/% 7]
head(ex)

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 3)
par(bg = "white")
with(ex[district_id == "ZAF_D01"], plot(week_start, tmax, type = "l", col = "grey30",
     xlab = "", ylab = "Weekly mean tmax (°C)", main = "ZAF_D01"))

## 6. Is heat related to preterm birth?

First the temperature anomaly: each district-week against that district's own mean for the same week of the year.

In [ ]:
ex[, woy := wk %% 52]
ex[, tmax_anom := tmax - mean(tmax), by = .(district_id, woy)]
setkey(ex, district_id, wk)

Then one row per pregnancy per gestational week from 22 to 36. `event` is 1 in the week the baby is born. About 2.7 million rows.

In [ ]:
p[, gw := floor(ga_days / 7)]
p[, lmp_day := as.integer(lmp_date - as.IDate("2100-01-01"))]
pw <- rbindlist(lapply(22:36, function(w)
  p[gw >= w, .(id, district_id, gest_week = w, event = as.integer(gw == w),
               week_start_day = lmp_day + 7L * w, age_years, ses_grade, hiv_positive,
               anaemia, hypertensive_disorder, outdoor_work_pregnancy, infant_sex)]))
nrow(pw)

Attach the anomaly for the current week and the four weeks before (lags 0 to 4).

In [ ]:
for (l in 0:4) {
  pw[, wk := (week_start_day - 1L - 7L * l) %/% 7L]
  pw[ex, (paste0("lag", l)) := i.tmax_anom, on = .(district_id, wk)]
}
pw <- na.omit(pw, cols = paste0("lag", 0:4))
pw[, c("wk", "week_start_day", "id", "district_id") := NULL]   # keep memory down for the models
cat(format(nrow(pw), big.mark = ","), "person-weeks,", format(sum(pw$event), big.mark = ","), "preterm births\n")

2.7 million rows is a lot for `glm` on a free Colab machine. Keep every birth and a random 1 in 5 of the weeks with no birth. Sampling at a fixed rate only shifts the intercept, so the odds ratios are unchanged (only a little less precise).

In [ ]:
set.seed(2026)
pw <- pw[event == 1 | runif(.N) < 0.2]
invisible(gc())
nrow(pw)

A simple model first: one lag-weighted heat term, adjusted for gestational week and the main risk factors.

In [ ]:
w <- c(0.40, 0.28, 0.17, 0.10, 0.05)
pw[, heat := w[1]*lag0 + w[2]*lag1 + w[3]*lag2 + w[4]*lag3 + w[5]*lag4]
pw[, `:=`(age_35plus = age_years > 35, age_under22 = age_years < 22)]

f <- event ~ factor(gest_week) + age_35plus + age_under22 + factor(ses_grade) + hiv_positive +
  anaemia + hypertensive_disorder + outdoor_work_pregnancy + infant_sex
m1 <- glm(update(f, ~ . + heat), family = binomial, data = pw, model = FALSE, y = FALSE)
round(exp(cbind(OR = coef(m1), confint.default(m1)))["heat", ], 3)   # per °C

## 7. A distributed lag model with `dlnm`

Let the data choose the lag shape. A linear exposure, with a natural spline over lags 0 to 4.

In [ ]:
cb <- crossbasis(as.matrix(pw[, paste0("lag", 0:4), with = FALSE]), lag = 4,
                 argvar = list(fun = "lin"), arglag = list(fun = "ns", df = 3))
rm(m1); invisible(gc())
m2 <- glm(update(f, ~ . + cb), family = binomial, data = pw, model = FALSE, y = FALSE)
pred <- crosspred(cb, m2, at = 1, cen = 0)
round(cbind(OR = pred$matRRfit[1, ], lo = pred$matRRlow[1, ], hi = pred$matRRhigh[1, ]), 3)   # per °C, by lag

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)
par(bg = "white")
plot(pred, "slices", var = 1, col = "grey20", ci.arg = list(col = "grey85"),
     xlab = "Lag (weeks)", ylab = "OR per °C", main = "Lag-response for a 1 °C anomaly")
cat("Cumulative OR over lags 0 to 4:", round(pred$allRRfit, 3),
    "(", round(pred$allRRlow, 3), "to", round(pred$allRRhigh, 3), ")\n")

## Your turn

The exposure here is the ready-made gridded product. Build your own temperature surface from the stations (`data/weather_stations_daily.parquet`, or the `.csv.gz` copy), average it to districts with `data/district_cells_0p25deg.csv`, and see whether the answer changes. Example results to beat are in `docs/method_comparison.md`.